**LangChain Parallel Chains**

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel
from pydantic import BaseModel, Field
from typing import List

In [4]:
class MovieInfo(BaseModel):
    movie: str = Field(..., description="Name of the movie")
    director: str = Field(..., description="Director of the movie")
    stars: List[str] = Field(..., description="List of stars in the movie")
    key_highlights: List[str] = Field(..., description="Key highlights of the movie")
    summary: str = Field(..., description="Summary of the movie")

**Main Chain**

In [5]:
#Task-1
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    ("user", "Hi ! give me summary about Movie {movie_name}")
])

#Task-2
llm_openai = init_chat_model(model="gpt-5-mini", model_provider="openai", temperature=0.1)
structured_llm_ouput = llm_openai.with_structured_output(MovieInfo)

#Task-4
def dictionary_maker(obj: MovieInfo) -> dict:
    
    return {"movie": obj.movie, "director": obj.director, "stars": obj.stars, "key_highlights": obj.key_highlights, "summary": obj.summary}

dictionary_generator = RunnableLambda(dictionary_maker)

**Parallel Branch 1**

In [11]:
def review_chain(movie_info: dict)-> str:
    review_template = ChatPromptTemplate.from_messages([
        ("system", "You are one of the best professional world famous movie critic. Give your rating and write professional movie review in IMDB"),
        ("human", "{movie}, {director}, {stars}, {key_highlights}, {summary}")
    ])

    llm_openai = init_chat_model(model="gpt-5-mini", model_provider="openai", temperature=0.2)
    output_parser = StrOutputParser()
    review_chain = review_template | llm_openai | output_parser
    return review_chain.invoke(movie_info)

review_chain_caller = RunnableLambda(review_chain)

**Parallel Branch 2**

In [12]:
def script_chain(movie_info: dict)-> str:
    script_template = ChatPromptTemplate.from_messages([
        ("system", "You are one of the best professional world famous movie script writer. you wrote numerous scripts across various genres in hollywood and bollywood. Write a professional movie sequel script for the given movie. It need not be detailed but short, concise, consistent and crisp. It should be a continuation of the original movie and should be in the same genre."),
        ("human", "{movie}, {director}, {stars}, {key_highlights}, {summary}")
    ])

    llm_openai = init_chat_model(model="gpt-5-mini", model_provider="openai", temperature=0.6)
    output_parser = StrOutputParser()
    script_chain = script_template | llm_openai | output_parser
    return script_chain.invoke(movie_info)

script_chain_caller = RunnableLambda(script_chain)

**Final Integration Chain**

In [14]:
final_Chain = prompt_template | structured_llm_ouput | dictionary_generator | RunnableParallel(branches={"review": review_chain_caller, "sequel_scope": script_chain_caller})
final_Chain.invoke({"movie_name": "Telugu Movie Darling (2010)"})

{'branches': {'review': 'Rating: 7/10\n\nDarling is a buoyant, crowd-pleasing Telugu romantic comedy that does exactly what it sets out to do: entertain. A. Karunakaran steers this glossy family entertainer with a sure hand, balancing light-hearted comedy, earnest sentiment and a streak of melodrama in roughly equal measure. The film’s familiar beats—the mistaken confidences, parental expectations, and sacrificial revelations—are predictable, but Karunakaran extracts genuine warmth and enough comic rhythm to keep the proceedings consistently engaging.\n\nPrabhas, as Siddhu, brings a winning, easy charm to the lead role. He’s affable and energetic, selling both the jokey moments and the film’s emotional demands with a naturalness that helped cement his image as a romantic hero. Kajal Aggarwal complements him well; she’s graceful, expressive and has a chemistry with Prabhas that feels effortless and central to the film’s appeal. The pair’s interactions are the film’s emotional anchor, an

In [1]:
def beautify_output(output: dict) -> dict:
    review = output['branches'].get('review')
    sequel_scope = output['branches'].get('sequel_scope')
    return {"review": review, "sequel_scope": sequel_scope}

beautify_output = RunnableLambda(beautify_output)

result_chain = final_Chain | beautify_output
result_chain.invoke({"movie_name": "Telugu Movie Irumudi (2026)"})

NameError: name 'RunnableLambda' is not defined